In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def create_bell_circuit():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

def test_bell_state_correlation(shots=2048, tolerance=0.05):
    """
    WHAT THIS TESTS:
    The Bell state must produce only correlated outcomes.
    Correlated fraction = P(00) + P(11) must be >= 0.95.
    Any '01' or '10' result on ideal simulator = circuit bug.

    WHY THIS MATTERS:
    This is the signature of quantum entanglement.
    A classical system cannot produce this correlation pattern.
    Testing it validates that the entanglement was created correctly.
    """

    counts = AerSimulator().run(
        create_bell_circuit(), shots=shots
    ).result().get_counts()

    total = sum(counts.values())
    p_corr = (counts.get('00', 0) + counts.get('11', 0)) / total

    passed = p_corr >= (1.0 - tolerance)
    status = "PASS" if passed else "FAIL"
    print(f"{status} | Bell State Correlation = {p_corr:.3f} | {counts}")
    return passed


def test_bell_no_anticorrelated_outcomes(shots=1024):
    """
    On ideal simulator: '01' and '10' must NEVER appear.
    Their presence = missing or wrong CNOT gate.
    """
    counts = AerSimulator().run(
        create_bell_circuit(), shots=shots
    ).result().get_counts()

    no_01 = counts.get('01', 0) == 0
    no_10 = counts.get('10', 0) == 0
    passed = no_01 and no_10

    print(f"{'PASS' if passed else 'FAIL'} | "
          f"No anticorrelated outcomes | {counts}")
    return passed

if __name__ == "__main__":
    print("Running Bell State Tests\n" + "="*40)
    r1 = test_bell_state_correlation()
    r2 = test_bell_no_anticorrelated_outcomes()
    print(f"\nFinal: {'ALL PASS ✓' if r1 and r2 else 'SOME FAILED ✗'}")


Running Bell State Tests
PASS | Bell State Correlation = 1.000 | {'11': 1030, '00': 1018}
PASS | No anticorrelated outcomes | {'00': 507, '11': 517}

Final: ALL PASS ✓
